In [17]:
import re
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 200)

RAW_PATH = Path("../data/raw/IST_corrected.csv")
UTF8_PATH = Path("../data/interim/IST_corrected_utf8.csv")

In [18]:
con = duckdb.connect()
con.sql("""
    CREATE VIEW ist AS
    SELECT * FROM read_csv_auto('../data/raw/IST_corrected.csv')
""")
con.sql("DESCRIBE ist").df()

InvalidInputException: Invalid Input Error: CSV Error on Line: 147
Original Line: 
5,46,F,M,52,N,,Y,Y,,,140,N,N,N,N,N,N,Y,N,POCS,wrz-91,99,99,2,N,N,N,Y,N,,N,3,,,N,N,N,N,N,N,,,N,,,N,,,Y,N,N,N,,N,,N,,,,N,,Y,3,,N,,0,,N,,,,,N,N,A,Y,N,29,185,PORT,19,,,Y,Y,0,185,0.1527,0.0223,0.0117,1,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Invalid unicode (byte sequence mismatch) detected. This file is not utf-8 encoded.

Possible Solution: Set the correct encoding, if available, to read this CSV File (e.g., encoding='UTF-16')
Possible Solution: Enable ignore errors (ignore_errors=true) to skip this row

  file = ../data/raw/IST_corrected.csv
  delimiter = , (Auto-Detected)
  quote = " (Auto-Detected)
  escape = " (Auto-Detected)
  new_line = Single-Line File (Auto-Detected)
  header = false (Auto-Detected)
  skip_rows = 0 (Auto-Detected)
  comment = (empty) (Auto-Detected)
  strict_mode = true (Auto-Detected)
  date_format =  (Auto-Detected)
  timestamp_format =  (Auto-Detected)
  null_padding = 0
  sample_s

In [19]:
raw = Path("../data/raw/IST_corrected.csv").read_bytes()

print("dimensione:", len(raw), "byte")
print("inizio file:", raw[:200])
print(
    "fine riga \\r\\n:",
    raw.count(b"\r\n"),
    " \\n:",
    raw.count(b"\n"),
    " \\r:",
    raw.count(b"\r"),
)

nonascii = sorted({b for b in raw if b > 127})
print("byte non-ASCII presenti:", [hex(b) for b in nonascii])

for m in list(re.finditer(rb"[\x80-\xff]+", raw))[:5]:
    print(raw[max(0, m.start() - 15) : m.end() + 15])

dimensione: 4798288 byte
inizio file: b'HOSPNUM,RDELAY,RCONSC,SEX,AGE,RSLEEP,RATRIAL,RCT,RVISINF,RHEP24,RASP3,RSBP,RDEF1,RDEF2,RDEF3,RDEF4,RDEF5,RDEF6,RDEF7,RDEF8,STYPE,RDATE,HOURLOCAL,MINLOCAL,DAYLOCAL,RXASP,RXHEP,DASP14,DASPLT,DLH14,DMH14'
fine riga \r\n: 19436  \n: 19436  \r: 19436
byte non-ASCII presenti: ['0x9f']
b'Y,Y,N,N,TACS,pa\x9f-91,99,99,3,Y,L'
b'N,N,N,N,PACS,pa\x9f-91,99,99,4,Y,L'
b'N,N,N,N,PACS,pa\x9f-91,99,99,4,N,N'
b'C,C,N,N,PACS,pa\x9f-91,99,99,5,N,H'
b'N,N,N,N,PACS,pa\x9f-91,99,99,5,Y,N'


In [20]:
text = raw.decode("cp1250")
assert "paź" in text, "decodifica inattesa: controllare l'encoding"

out = Path("../data/interim/IST_corrected_utf8.csv")
out.write_bytes(text.encode("utf-8"))
print("scritto:", out, out.stat().st_size, "byte")

scritto: ..\data\interim\IST_corrected_utf8.csv 4799920 byte


In [21]:
con.sql("""
    CREATE OR REPLACE VIEW ist AS
    SELECT * FROM read_csv_auto('../data/interim/IST_corrected_utf8.csv')
""")
print(con.sql("SELECT COUNT(*) AS n FROM ist").fetchone())
con.sql("DESCRIBE ist").df()

(19435,)


,column_name,column_type,null,key,default,extra
0,HOSPNUM,BIGINT,YES,None,None,None
1,RDELAY,BIGINT,YES,None,None,None
2,RCONSC,VARCHAR,YES,None,None,None
3,SEX,VARCHAR,YES,None,None,None
4,AGE,BIGINT,YES,None,None,None
5,RSLEEP,VARCHAR,YES,None,None,None
6,RATRIAL,VARCHAR,YES,None,None,None
7,RCT,VARCHAR,YES,None,None,None
8,RVISINF,VARCHAR,YES,None,None,None
9,RHEP24,VARCHAR,YES,None,None,None


In [9]:
con = duckdb.connect()
con.sql(f"""
    CREATE OR REPLACE VIEW ist AS
    SELECT * FROM read_csv_auto('{UTF8_PATH.as_posix()}')
""")
print(con.sql("SELECT COUNT(*) AS n FROM ist").fetchone())
con.sql("DESCRIBE ist").df()

(19435,)


,column_name,column_type,null,key,default,extra
0,HOSPNUM,BIGINT,YES,None,None,None
1,RDELAY,BIGINT,YES,None,None,None
2,RCONSC,VARCHAR,YES,None,None,None
3,SEX,VARCHAR,YES,None,None,None
4,AGE,BIGINT,YES,None,None,None
...,...,...,...,...,...,...
107,HTI14,BIGINT,YES,None,None,None
108,PE14,BIGINT,YES,None,None,None
109,DVT14,BIGINT,YES,None,None,None
110,TRAN14,BIGINT,YES,None,None,None


In [10]:
import pandas as pd

pd.set_option("display.max_rows", 200)
con.sql("DESCRIBE ist").df()

,column_name,column_type,null,key,default,extra
0,HOSPNUM,BIGINT,YES,None,None,None
1,RDELAY,BIGINT,YES,None,None,None
2,RCONSC,VARCHAR,YES,None,None,None
3,SEX,VARCHAR,YES,None,None,None
4,AGE,BIGINT,YES,None,None,None
5,RSLEEP,VARCHAR,YES,None,None,None
6,RATRIAL,VARCHAR,YES,None,None,None
7,RCT,VARCHAR,YES,None,None,None
8,RVISINF,VARCHAR,YES,None,None,None
9,RHEP24,VARCHAR,YES,None,None,None


In [22]:
con.sql("""
    SELECT column_type, COUNT(*) AS n_columns
    FROM (DESCRIBE ist)
    GROUP BY column_type
    ORDER BY n_columns DESC
""").df()

,column_type,n_columns
0,VARCHAR,63
1,BIGINT,46
2,DOUBLE,3


In [23]:
con.sql("SUMMARIZE ist").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,HOSPNUM,BIGINT,1,568,607,200.05546694108568,160.33509345877553,61,158,320,19435,0.00
1,RDELAY,BIGINT,1,48,45,20.124363262155903,12.470329949601897,9,19,29,19435,0.00
2,RCONSC,VARCHAR,D,U,3,NaN,NaN,NaN,NaN,NaN,19435,0.00
3,SEX,VARCHAR,F,M,2,NaN,NaN,NaN,NaN,NaN,19435,0.00
4,AGE,BIGINT,16,99,78,71.7154103421662,11.61971381005605,65,73,80,19435,0.00
5,RSLEEP,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,19435,0.00
6,RATRIAL,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,19435,5.06
7,RCT,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,19435,0.00
8,RVISINF,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,19435,0.00
9,RHEP24,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,19435,1.77


In [14]:
schema = con.sql("DESCRIBE ist").df()
schema["column_type"]

0       BIGINT
1       BIGINT
2      VARCHAR
3      VARCHAR
4       BIGINT
5      VARCHAR
6      VARCHAR
7      VARCHAR
8      VARCHAR
9      VARCHAR
10     VARCHAR
11      BIGINT
12     VARCHAR
13     VARCHAR
14     VARCHAR
15     VARCHAR
16     VARCHAR
17     VARCHAR
18     VARCHAR
19     VARCHAR
20     VARCHAR
21     VARCHAR
22      BIGINT
23      BIGINT
24      BIGINT
25     VARCHAR
26     VARCHAR
27     VARCHAR
28     VARCHAR
29     VARCHAR
30     VARCHAR
31     VARCHAR
32      BIGINT
33     VARCHAR
34     VARCHAR
35     VARCHAR
36     VARCHAR
37     VARCHAR
38     VARCHAR
39     VARCHAR
40     VARCHAR
41     VARCHAR
42     VARCHAR
43     VARCHAR
44      BIGINT
45     VARCHAR
46     VARCHAR
47      BIGINT
48     VARCHAR
49     VARCHAR
50     VARCHAR
51     VARCHAR
52     VARCHAR
53     VARCHAR
54     VARCHAR
55      BIGINT
56     VARCHAR
57      BIGINT
58     VARCHAR
59      BIGINT
60     VARCHAR
61      BIGINT
62     VARCHAR
63      BIGINT
64     VARCHAR
65     VARCHAR
66      BI

In [15]:
schema["column_type"].value_counts()

column_type
VARCHAR    63
BIGINT     46
DOUBLE      3
Name: count, dtype: int64

In [16]:
con.sql("SUMMARIZE ist").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,HOSPNUM,BIGINT,1,568,607,200.05546694108568,160.33509345877553,61,158,320,19435,0.00
1,RDELAY,BIGINT,1,48,45,20.124363262155903,12.470329949601897,9,19,29,19435,0.00
2,RCONSC,VARCHAR,D,U,3,NaN,NaN,NaN,NaN,NaN,19435,0.00
3,SEX,VARCHAR,F,M,2,NaN,NaN,NaN,NaN,NaN,19435,0.00
4,AGE,BIGINT,16,99,78,71.7154103421662,11.61971381005605,65,73,80,19435,0.00
5,RSLEEP,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,19435,0.00
6,RATRIAL,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,19435,5.06
7,RCT,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,19435,0.00
8,RVISINF,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,19435,0.00
9,RHEP24,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,19435,1.77
